Please use this notebook to perform any initial analyses and model testing!

In [0]:
import pandas as pd
import numpy as np
import os 
import scipy

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
def create_or_update_unity_table(catalog, schema, table, df):
    spark = SparkSession.builder.getOrCreate()

    create_schema_sql = f"""
        CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}
    """
    
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table}
    """
    spark.sql(create_schema_sql)
    spark.sql(create_table_sql)
    spark.createDataFrame(df).write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{catalog}.{schema}.{table}")
    print(f"Table {catalog}.{schema}.{table} created or updated.")


In [0]:
df_bronze = pd.read_csv("/Workspace/Users/corey.bevan@gmail.com/wind_turbine_drift_detection/data/bronze/bronze.csv").drop(columns = ['Unnamed: 0'])

display(df_bronze)

In [0]:
df_bronze.columns = ['timestamp', 'lv_active_power', 'wind_speed', 'theoretical_power_curve', 'wind_direction']

create_or_update_unity_table("workspace", "ds_cb_wind_turbine", "bronze", df_bronze)
df_bronze.head()

In [0]:
print(df_bronze.columns)

In [0]:
df_bronze['timestamp'] = pd.to_datetime(df_bronze['timestamp'], format='%d %m %Y %H:%M')
print(df_bronze[df_bronze['timestamp'] == pd.NaT])
df_bronze.head()

In [0]:
print(df_bronze.loc[df_bronze['lv_active_power'] < 0]['lv_active_power'].describe())

In [0]:
df_bronze['lv_active_power'] = df_bronze['lv_active_power'].clip(lower = 0)

In [0]:
df_bronze['hour'] = pd.to_datetime(df_bronze['timestamp']).dt.hour
df_bronze['monr'] = pd.to_datetime(df_bronze['timestamp']).dt.hour

In [0]:
print(df_bronze['timestamp'].min())
print(df_bronze['timestamp'].max())

In [0]:
df_bronze.to_csv("../data/silver/silver.csv")
create_or_update_unity_table("workspace", "ds_cb_wind_turbine", "silver", df_bronze)

In [0]:
df_bronze.head()

In [0]:
df_bronze['sine_hour'] = np.sin(2 * np.pi * df_bronze['hour']/24)
df_bronze['cosine_hour'] = np.cos(2 * np.pi * df_bronze['hour']/24)

df_bronze = df_bronze.sort_values(by = ['timestamp'])

In [0]:
df_bronze['wind_direction_rad'] = np.deg2rad(df_bronze['wind_direction'])
df_bronze['wind_magnitude_i'] = df_bronze['wind_speed'] * (np.cos(df_bronze['wind_direction_rad']))
df_bronze['wind_magnitude_j'] = df_bronze['wind_speed'] * (np.sin(df_bronze['wind_direction_rad']))

In [0]:
df_bronze['wind_magnitude_i_3h_rolling'] = df_bronze['wind_magnitude_i'].rolling(window = 3).mean()
df_bronze['wind_magnitude_j_3h_rolling'] = df_bronze['wind_magnitude_j'].rolling(window = 3).mean()
df_bronze['wind_speed_3h_rolling'] = df_bronze['wind_speed'].rolling(window = 3).mean()
df_bronze.dropna(inplace=True)

In [0]:
df_bronze.to_csv("../data/gold/gold.csv")
create_or_update_unity_table("workspace", "ds_cb_wind_turbine", "gold", df_bronze)